In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Product-Analytics-Project/user-level-data.csv')
display(df.head())

,user_id,platform,region,ab_cohort_name,total_revenue,returned_d1,total_sessions,avg_session_length,median_session_length,total_fs_shown,total_rv_shown
0,f7c1a568-c994-4876-85be-7bf1fb84dd4c,android,Asia-Pacific,xxLow,0.668772,1,48,174.06,83,75,3
1,7f4a3cbc-892d-4f13-b28c-c95f2ad72644,android,Asia-Pacific,xxLow,0.153658,1,14,270.50,215,38,0
2,2f918da8-24ab-412f-bdee-7284c096b774,android,Asia-Pacific,xxLow,0.121999,1,15,140.87,132,9,0
3,f9c92b6a-b008-41fb-bfe2-4b05df6a84a3,android,Asia-Pacific,xxLow,0.374711,1,14,243.79,240,35,0
4,7840771b-2ffc-424b-a53e-1ccddc8b6154,android,LATAM,xxLow,0.167829,1,16,474.69,221,108,1


In [8]:
print(f"Total users: {len(df):,}")
print(f"Cohorts: {df['ab_cohort_name'].unique()}")
print(f"Platforms: {df['platform'].unique()}")
print(f"Regions: {df['region'].unique()}")
print()

Total users: 100,129
Cohorts: ['xxLow' 'xLow' 'control' 'xHigh' 'xxHigh' 'gameTune']
Platforms: ['android' 'ios']
Regions: ['Asia-Pacific' 'LATAM' 'North America' 'Europe'
 'MEA (Middle East & Africa)']



In [3]:
# ============================================
# 2. HELPER FUNCTIONS
# ============================================

def ttest_cohort_vs_control(df, metric, cohort, control='control',
                            platform=None, region=None):
    """
    Perform t-test for a continuous metric between cohort and control.
    Returns: mean_control, mean_cohort, delta, delta_pct, p_value, significant
    """
    # Filter data
    df_filtered = df.copy()
    if platform:
        df_filtered = df_filtered[df_filtered['platform'] == platform]
    if region:
        df_filtered = df_filtered[df_filtered['region'] == region]

    control_data = df_filtered[df_filtered['ab_cohort_name'] == control][metric].dropna()
    cohort_data = df_filtered[df_filtered['ab_cohort_name'] == cohort][metric].dropna()

    if len(control_data) < 30 or len(cohort_data) < 30:
        return None  # Not enough data

    # T-test
    t_stat, p_value = stats.ttest_ind(cohort_data, control_data, equal_var=False)

    mean_control = control_data.mean()
    mean_cohort = cohort_data.mean()
    delta = mean_cohort - mean_control
    delta_pct = (delta / mean_control * 100) if mean_control != 0 else 0
    significant = p_value < 0.05

    # Confidence interval for difference
    se = np.sqrt(control_data.var()/len(control_data) + cohort_data.var()/len(cohort_data))
    ci_lower = delta - 1.96 * se
    ci_upper = delta + 1.96 * se

    return {
        'control_mean': mean_control,
        'cohort_mean': mean_cohort,
        'delta': delta,
        'delta_pct': delta_pct,
        'p_value': p_value,
        'significant': significant,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'control_n': len(control_data),
        'cohort_n': len(cohort_data)
    }


def proptest_cohort_vs_control(df, cohort, control='control',
                               platform=None, region=None):
    """
    Perform proportion z-test for D1 retention between cohort and control.
    Returns: prop_control, prop_cohort, delta_pp, p_value, significant
    """
    # Filter data
    df_filtered = df.copy()
    if platform:
        df_filtered = df_filtered[df_filtered['platform'] == platform]
    if region:
        df_filtered = df_filtered[df_filtered['region'] == region]

    control_data = df_filtered[df_filtered['ab_cohort_name'] == control]['returned_d1'].dropna()
    cohort_data = df_filtered[df_filtered['ab_cohort_name'] == cohort]['returned_d1'].dropna()

    if len(control_data) < 30 or len(cohort_data) < 30:
        return None

    # Count successes
    count_control = control_data.sum()
    count_cohort = cohort_data.sum()
    nobs_control = len(control_data)
    nobs_cohort = len(cohort_data)

    # Z-test for proportions
    counts = np.array([count_cohort, count_control])
    nobs = np.array([nobs_cohort, nobs_control])

    z_stat, p_value = proportions_ztest(counts, nobs)

    prop_control = count_control / nobs_control
    prop_cohort = count_cohort / nobs_cohort
    delta_pp = (prop_cohort - prop_control) * 100  # percentage points
    significant = p_value < 0.05

    # Confidence interval for difference in proportions
    se = np.sqrt(prop_control*(1-prop_control)/nobs_control +
                 prop_cohort*(1-prop_cohort)/nobs_cohort)
    ci_lower = (prop_cohort - prop_control - 1.96*se) * 100
    ci_upper = (prop_cohort - prop_control + 1.96*se) * 100

    return {
        'control_prop': prop_control * 100,
        'cohort_prop': prop_cohort * 100,
        'delta_pp': delta_pp,
        'p_value': p_value,
        'significant': significant,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'control_n': nobs_control,
        'cohort_n': nobs_cohort
    }


In [4]:
# ============================================
# 3. GLOBAL ANALYSIS (ALL USERS)
# ============================================

print("=" * 60)
print("GLOBAL ANALYSIS: ARPU vs Control")
print("=" * 60)

cohorts_to_test = ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']

arpu_results_global = []
for cohort in cohorts_to_test:
    result = ttest_cohort_vs_control(df, 'total_revenue', cohort)
    if result:
        arpu_results_global.append({
            'cohort': cohort,
            'control_arpu': f"${result['control_mean']:.6f}",
            'cohort_arpu': f"${result['cohort_mean']:.6f}",
            'delta': f"${result['delta']:.6f}",
            'delta_pct': f"{result['delta_pct']:+.2f}%",
            'p_value': f"{result['p_value']:.4f}",
            'significant': '✓' if result['significant'] else '✗',
            'ci_95': f"[{result['ci_lower']:.6f}, {result['ci_upper']:.6f}]"
        })

df_arpu_global = pd.DataFrame(arpu_results_global)
print(df_arpu_global.to_string(index=False))
print()

print("=" * 60)
print("GLOBAL ANALYSIS: D1 Retention vs Control")
print("=" * 60)

d1_results_global = []
for cohort in cohorts_to_test:
    result = proptest_cohort_vs_control(df, cohort)
    if result:
        d1_results_global.append({
            'cohort': cohort,
            'control_d1': f"{result['control_prop']:.2f}%",
            'cohort_d1': f"{result['cohort_prop']:.2f}%",
            'delta_pp': f"{result['delta_pp']:+.2f} pp",
            'p_value': f"{result['p_value']:.4f}",
            'significant': '✓' if result['significant'] else '✗',
            'ci_95': f"[{result['ci_lower']:.2f}, {result['ci_upper']:.2f}] pp"
        })

df_d1_global = pd.DataFrame(d1_results_global)
print(df_d1_global.to_string(index=False))
print()


GLOBAL ANALYSIS: ARPU vs Control
  cohort control_arpu cohort_arpu      delta delta_pct p_value significant                  ci_95
  xxHigh    $0.143681   $0.149581  $0.005900    +4.11%  0.0197           ✓   [0.000943, 0.010858]
   xHigh    $0.143681   $0.148212  $0.004531    +3.15%  0.0700           ✗  [-0.000371, 0.009434]
gameTune    $0.143681   $0.143673 $-0.000008    -0.01%  0.9975           ✗  [-0.004836, 0.004821]
    xLow    $0.143681   $0.127508 $-0.016173   -11.26%  0.0000           ✓ [-0.020697, -0.011648]
   xxLow    $0.143681   $0.123179 $-0.020502   -14.27%  0.0000           ✓ [-0.024993, -0.016011]

GLOBAL ANALYSIS: D1 Retention vs Control
  cohort control_d1 cohort_d1 delta_pp p_value significant            ci_95
  xxHigh     32.77%    32.59% -0.18 pp  0.7264           ✗ [-1.19, 0.83] pp
   xHigh     32.77%    32.32% -0.44 pp  0.3872           ✗ [-1.45, 0.56] pp
gameTune     32.77%    32.33% -0.44 pp  0.3918           ✗ [-1.45, 0.57] pp
    xLow     32.77%    33.40% +0.

In [7]:
print("=" * 60)
print("PLATFORM ANALYSIS: ARPU by Platform (all cohorts vs Control)")
print("=" * 60)

platforms = ['android', 'ios']
cohorts_to_test = ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']

arpu_results_platform = []

for platform in platforms:
    for cohort in cohorts_to_test:
        result = ttest_cohort_vs_control(df, 'total_revenue', cohort, platform=platform)
        if result:
            arpu_results_platform.append({
                'platform': platform,
                'cohort': cohort,
                'control_arpu': f"{result['control_mean']:.6f}",
                'cohort_arpu': f"{result['cohort_mean']:.6f}",
                'delta': f"{result['delta']:.6f}",
                'delta_pct': f"{result['delta_pct']:+.2f}%",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_arpu_platform = pd.DataFrame(arpu_results_platform)
print(df_arpu_platform.to_string(index=False))
print()

print("=" * 60)
print("PLATFORM ANALYSIS: D1 Retention by Platform (all cohorts vs Control)")
print("=" * 60)

d1_results_platform = []

for platform in platforms:
    for cohort in cohorts_to_test:
        result = proptest_cohort_vs_control(df, cohort, platform=platform)
        if result:
            d1_results_platform.append({
                'platform': platform,
                'cohort': cohort,
                'control_d1': f"{result['control_prop']:.2f}%",
                'cohort_d1': f"{result['cohort_prop']:.2f}%",
                'delta_pp': f"{result['delta_pp']:+.2f} pp",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_d1_platform = pd.DataFrame(d1_results_platform)
print(df_d1_platform.to_string(index=False))
print()


PLATFORM ANALYSIS: ARPU by Platform (all cohorts vs Control)
platform   cohort control_arpu cohort_arpu     delta delta_pct p_value significant
 android   xxHigh     0.104852    0.110054  0.005202    +4.96%  0.0155           ✓
 android    xHigh     0.104852    0.106488  0.001636    +1.56%  0.4235           ✗
 android gameTune     0.104852    0.104741 -0.000111    -0.11%  0.9563           ✗
 android     xLow     0.104852    0.096510 -0.008342    -7.96%  0.0000           ✓
 android    xxLow     0.104852    0.093779 -0.011073   -10.56%  0.0000           ✓
     ios   xxHigh     0.240086    0.251982  0.011895    +4.95%  0.0743           ✗
     ios    xHigh     0.240086    0.256143  0.016057    +6.69%  0.0169           ✓
     ios gameTune     0.240086    0.242998  0.002912    +1.21%  0.6579           ✗
     ios     xLow     0.240086    0.207644 -0.032443   -13.51%  0.0000           ✓
     ios    xxLow     0.240086    0.199940 -0.040146   -16.72%  0.0000           ✓

PLATFORM ANALYSIS: D1 Ret

In [8]:
# ============================================
# 5. REGION ANALYSIS
# ============================================

print("=" * 60)
print("REGION ANALYSIS: ARPU by Region (all cohorts vs Control)")
print("=" * 60)

regions = [r for r in df['region'].unique() if pd.notnull(r)]
cohorts_to_test = ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']

arpu_results_region = []

for region in regions:
    for cohort in cohorts_to_test:
        result = ttest_cohort_vs_control(df, 'total_revenue', cohort, region=region)
        if result:
            arpu_results_region.append({
                'region': region,
                'cohort': cohort,
                'control_arpu': f"{result['control_mean']:.6f}",
                'cohort_arpu': f"{result['cohort_mean']:.6f}",
                'delta': f"{result['delta']:.6f}",
                'delta_pct': f"{result['delta_pct']:+.2f}%",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_arpu_region = pd.DataFrame(arpu_results_region)
print(df_arpu_region.to_string(index=False))
print()

print("=" * 60)
print("REGION ANALYSIS: D1 Retention by Region (all cohorts vs Control)")
print("=" * 60)

d1_results_region = []

for region in regions:
    for cohort in cohorts_to_test:
        result = proptest_cohort_vs_control(df, cohort, region=region)
        if result:
            d1_results_region.append({
                'region': region,
                'cohort': cohort,
                'control_d1': f"{result['control_prop']:.2f}%",
                'cohort_d1': f"{result['cohort_prop']:.2f}%",
                'delta_pp': f"{result['delta_pp']:+.2f} pp",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_d1_region = pd.DataFrame(d1_results_region)
print(df_d1_region.to_string(index=False))
print()


REGION ANALYSIS: ARPU by Region (all cohorts vs Control)
                    region   cohort control_arpu cohort_arpu     delta delta_pct p_value significant
              Asia-Pacific   xxHigh     0.098316    0.101015  0.002699    +2.75%  0.7104           ✗
              Asia-Pacific    xHigh     0.098316    0.093538 -0.004778    -4.86%  0.4835           ✗
              Asia-Pacific gameTune     0.098316    0.088245 -0.010071   -10.24%  0.1221           ✗
              Asia-Pacific     xLow     0.098316    0.083090 -0.015226   -15.49%  0.0148           ✓
              Asia-Pacific    xxLow     0.098316    0.083394 -0.014922   -15.18%  0.0153           ✓
                     LATAM   xxHigh     0.026544    0.027043  0.000498    +1.88%  0.8509           ✗
                     LATAM    xHigh     0.026544    0.024768 -0.001776    -6.69%  0.4129           ✗
                     LATAM gameTune     0.026544    0.024484 -0.002061    -7.76%  0.3833           ✗
                     LATAM     xLo

In [9]:
# ============================================
# 6. ENGAGEMENT GUARDRAILS: SESSION LENGTH ANALYSIS
# ============================================

def mannwhitney_cohort_vs_control(df, metric, cohort, control='control',
                                  platform=None, region=None):
    """
    Perform Mann-Whitney U test for median comparison between cohort and control.
    Returns: median_control, median_cohort, delta, delta_pct, p_value, significant
    """
    # Filter data
    df_filtered = df.copy()
    if platform:
        df_filtered = df_filtered[df_filtered['platform'] == platform]
    if region:
        df_filtered = df_filtered[df_filtered['region'] == region]

    control_data = df_filtered[df_filtered['ab_cohort_name'] == control][metric].dropna()
    cohort_data = df_filtered[df_filtered['ab_cohort_name'] == cohort][metric].dropna()

    if len(control_data) < 30 or len(cohort_data) < 30:
        return None

    # Mann-Whitney U test
    u_stat, p_value = stats.mannwhitneyu(cohort_data, control_data, alternative='two-sided')

    median_control = control_data.median()
    median_cohort = cohort_data.median()
    delta = median_cohort - median_control
    delta_pct = (delta / median_control * 100) if median_control != 0 else 0
    significant = p_value < 0.05

    return {
        'control_median': median_control,
        'cohort_median': median_cohort,
        'delta': delta,
        'delta_pct': delta_pct,
        'p_value': p_value,
        'significant': significant,
        'control_n': len(control_data),
        'cohort_n': len(cohort_data)
    }

In [10]:
# ============================================
# 6A. ENGAGEMENT GUARDRAILS: GLOBAL ENGAGEMENT ANALYSIS
# ============================================

print("=" * 60)
print("ENGAGEMENT ANALYSIS (GLOBAL): Average Session Length vs Control")
print("=" * 60)

cohorts_to_test = ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']
avg_session_results_global = []

for cohort in cohorts_to_test:
    result = ttest_cohort_vs_control(df, 'avg_session_length', cohort)
    if result:
        avg_session_results_global.append({
            'cohort': cohort,
            'control_avg': f"{result['control_mean']:.2f}s",
            'cohort_avg': f"{result['cohort_mean']:.2f}s",
            'delta': f"{result['delta']:.2f}s",
            'delta_pct': f"{result['delta_pct']:+.2f}%",
            'p_value': f"{result['p_value']:.4f}",
            'significant': '✓' if result['significant'] else '✗'
        })

df_avg_session_global = pd.DataFrame(avg_session_results_global)
print(df_avg_session_global.to_string(index=False))
print()

print("=" * 60)
print("ENGAGEMENT ANALYSIS (GLOBAL): Median Session Length vs Control")
print("=" * 60)

median_session_results_global = []

for cohort in cohorts_to_test:
    result = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort)
    if result:
        median_session_results_global.append({
            'cohort': cohort,
            'control_median': f"{result['control_median']:.2f}s",
            'cohort_median': f"{result['cohort_median']:.2f}s",
            'delta': f"{result['delta']:.2f}s",
            'delta_pct': f"{result['delta_pct']:+.2f}%",
            'p_value': f"{result['p_value']:.4f}",
            'significant': '✓' if result['significant'] else '✗'
        })

df_median_session_global = pd.DataFrame(median_session_results_global)
print(df_median_session_global.to_string(index=False))
print()

ENGAGEMENT ANALYSIS (GLOBAL): Average Session Length vs Control
  cohort control_avg cohort_avg   delta delta_pct p_value significant
  xxHigh     351.60s    340.24s -11.36s    -3.23%  0.0018           ✓
   xHigh     351.60s    353.82s   2.22s    +0.63%  0.5522           ✗
gameTune     351.60s    343.91s  -7.68s    -2.18%  0.0348           ✓
    xLow     351.60s    368.95s  17.36s    +4.94%  0.0000           ✓
   xxLow     351.60s    376.92s  25.32s    +7.20%  0.0000           ✓

ENGAGEMENT ANALYSIS (GLOBAL): Median Session Length vs Control
  cohort control_median cohort_median  delta delta_pct p_value significant
  xxHigh        179.00s       170.00s -9.00s    -5.03%  0.0023           ✓
   xHigh        179.00s       178.00s -1.00s    -0.56%  0.8356           ✗
gameTune        179.00s       173.00s -6.00s    -3.35%  0.0237           ✓
    xLow        179.00s       191.00s 12.00s    +6.70%  0.0000           ✓
   xxLow        179.00s       195.00s 16.00s    +8.94%  0.0000           ✓



In [11]:
# ============================================
# 6B. PLATFORM ENGAGEMENT ANALYSIS
# ============================================

print("=" * 60)
print("ENGAGEMENT ANALYSIS (PLATFORM): Average Session Length")
print("=" * 60)

platforms = ['android', 'ios']
avg_session_results_platform = []

for platform in platforms:
    for cohort in cohorts_to_test:
        result = ttest_cohort_vs_control(df, 'avg_session_length', cohort, platform=platform)
        if result:
            avg_session_results_platform.append({
                'platform': platform,
                'cohort': cohort,
                'control_avg': f"{result['control_mean']:.2f}s",
                'cohort_avg': f"{result['cohort_mean']:.2f}s",
                'delta_pct': f"{result['delta_pct']:+.2f}%",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_avg_session_platform = pd.DataFrame(avg_session_results_platform)
print(df_avg_session_platform.to_string(index=False))
print()

print("=" * 60)
print("ENGAGEMENT ANALYSIS (PLATFORM): Median Session Length")
print("=" * 60)

median_session_results_platform = []

for platform in platforms:
    for cohort in cohorts_to_test:
        result = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort, platform=platform)
        if result:
            median_session_results_platform.append({
                'platform': platform,
                'cohort': cohort,
                'control_median': f"{result['control_median']:.2f}s",
                'cohort_median': f"{result['cohort_median']:.2f}s",
                'delta_pct': f"{result['delta_pct']:+.2f}%",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_median_session_platform = pd.DataFrame(median_session_results_platform)
print(df_median_session_platform.to_string(index=False))
print()


ENGAGEMENT ANALYSIS (PLATFORM): Average Session Length
platform   cohort control_avg cohort_avg delta_pct p_value significant
 android   xxHigh     373.54s    364.43s    -2.44%  0.0469           ✓
 android    xHigh     373.54s    379.41s    +1.57%  0.2128           ✗
 android gameTune     373.54s    367.13s    -1.72%  0.1603           ✗
 android     xLow     373.54s    391.59s    +4.83%  0.0001           ✓
 android    xxLow     373.54s    397.39s    +6.38%  0.0000           ✓
     ios   xxHigh     297.11s    277.56s    -6.58%  0.0003           ✓
     ios    xHigh     297.11s    287.63s    -3.19%  0.0832           ✗
     ios gameTune     297.11s    284.68s    -4.18%  0.0231           ✓
     ios     xLow     297.11s    310.42s    +4.48%  0.0174           ✓
     ios    xxLow     297.11s    323.47s    +8.87%  0.0000           ✓

ENGAGEMENT ANALYSIS (PLATFORM): Median Session Length
platform   cohort control_median cohort_median delta_pct p_value significant
 android   xxHigh        195.00s

In [12]:
# ============================================
# 6C. REGION ENGAGEMENT ANALYSIS
# ============================================

print("=" * 60)
print("ENGAGEMENT ANALYSIS (REGION): Average Session Length")
print("=" * 60)

regions = [r for r in df['region'].unique() if pd.notnull(r)]
avg_session_results_region = []

for region in regions:
    for cohort in cohorts_to_test:
        result = ttest_cohort_vs_control(df, 'avg_session_length', cohort, region=region)
        if result:
            avg_session_results_region.append({
                'region': region,
                'cohort': cohort,
                'control_avg': f"{result['control_mean']:.2f}s",
                'cohort_avg': f"{result['cohort_mean']:.2f}s",
                'delta_pct': f"{result['delta_pct']:+.2f}%",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_avg_session_region = pd.DataFrame(avg_session_results_region)
print(df_avg_session_region.to_string(index=False))
print()

print("=" * 60)
print("ENGAGEMENT ANALYSIS (REGION): Median Session Length")
print("=" * 60)

median_session_results_region = []

for region in regions:
    for cohort in cohorts_to_test:
        result = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort, region=region)
        if result:
            median_session_results_region.append({
                'region': region,
                'cohort': cohort,
                'control_median': f"{result['control_median']:.2f}s",
                'cohort_median': f"{result['cohort_median']:.2f}s",
                'delta_pct': f"{result['delta_pct']:+.2f}%",
                'p_value': f"{result['p_value']:.4f}",
                'significant': '✓' if result['significant'] else '✗'
            })

df_median_session_region = pd.DataFrame(median_session_results_region)
print(df_median_session_region.to_string(index=False))
print()

ENGAGEMENT ANALYSIS (REGION): Average Session Length
                    region   cohort control_avg cohort_avg delta_pct p_value significant
              Asia-Pacific   xxHigh     359.65s    334.42s    -7.01%  0.0559           ✗
              Asia-Pacific    xHigh     359.65s    350.24s    -2.62%  0.4746           ✗
              Asia-Pacific gameTune     359.65s    329.85s    -8.29%  0.0198           ✓
              Asia-Pacific     xLow     359.65s    369.24s    +2.67%  0.5196           ✗
              Asia-Pacific    xxLow     359.65s    376.74s    +4.75%  0.2079           ✗
                     LATAM   xxHigh     346.33s    349.59s    +0.94%  0.8003           ✗
                     LATAM    xHigh     346.33s    361.44s    +4.36%  0.2471           ✗
                     LATAM gameTune     346.33s    356.53s    +2.94%  0.4330           ✗
                     LATAM     xLow     346.33s    366.57s    +5.84%  0.1140           ✗
                     LATAM    xxLow     346.33s    380.48

In [16]:
# ============================================
# 7. DECISION FRAMEWORK APPLICATION
# ============================================

print("=" * 80)
print("DECISION FRAMEWORK: Revised Tiered Thresholds")
print("=" * 80)
print()
print("THRESHOLDS:")
print("  • Monetization: ARPU uplift ≥ +3.0% AND p < 0.10")
print("  • D1 Retention:")
print("      - If NOT significant (p ≥ 0.05): PASS automatically")
print("      - If significant (p < 0.05): Drop must be ≤ 1.0 pp")
print("  • Avg Session Length:")
print("      - If NOT significant (p ≥ 0.05): PASS automatically")
print("      - If significant (p < 0.05): Drop must be ≤ 3.0%")
print("  • Median Session Length:")
print("      - If NOT significant (p ≥ 0.05): PASS automatically")
print("      - If significant (p < 0.05): Drop must be ≤ 4.0%")
print()
print("Rationale: We distinguish between 'no proven effect' (non-sig) and")
print("           'proven effect that must be acceptable' (sig with strict limits)")
print()
print("=" * 80)

DECISION FRAMEWORK: Revised Tiered Thresholds

THRESHOLDS:
  • Monetization: ARPU uplift ≥ +3.0% AND p < 0.10
  • D1 Retention:
      - If NOT significant (p ≥ 0.05): PASS automatically
      - If significant (p < 0.05): Drop must be ≤ 1.0 pp
  • Avg Session Length:
      - If NOT significant (p ≥ 0.05): PASS automatically
      - If significant (p < 0.05): Drop must be ≤ 3.0%
  • Median Session Length:
      - If NOT significant (p ≥ 0.05): PASS automatically
      - If significant (p < 0.05): Drop must be ≤ 4.0%

Rationale: We distinguish between 'no proven effect' (non-sig) and
           'proven effect that must be acceptable' (sig with strict limits)



In [17]:
# ============================================
# 7A. GLOBAL WINNER EVALUATION
# ============================================

print()
print("=" * 80)
print("GLOBAL EVALUATION")
print("=" * 80)
print()

candidates = ['xxHigh', 'xHigh', 'gameTune']
global_results = []

for cohort in candidates:
    print(f"Evaluating {cohort} vs Control:")
    print("-" * 80)

    # Get test results
    arpu_test = ttest_cohort_vs_control(df, 'total_revenue', cohort)
    d1_test = proptest_cohort_vs_control(df, cohort)
    avg_session_test = ttest_cohort_vs_control(df, 'avg_session_length', cohort)
    median_session_test = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort)

    # 1. Monetization check (relaxed to p < 0.10)
    print(f"1. ARPU: {arpu_test['cohort_mean']:.6f} vs {arpu_test['control_mean']:.6f}")
    print(f"   Delta: {arpu_test['delta_pct']:+.2f}% (p={arpu_test['p_value']:.4f})")

    arpu_pass = (arpu_test['p_value'] < 0.10) and (arpu_test['delta_pct'] >= 3.0)
    if arpu_pass:
        sig_level = 'p<0.05' if arpu_test['p_value'] < 0.05 else '0.05≤p<0.10'
        print(f"   ✅ PASS: Uplift ≥ +3% and {sig_level}")
    else:
        if arpu_test['p_value'] >= 0.10:
            print(f"   ❌ FAIL: Not significant (p ≥ 0.10)")
        else:
            print(f"   ❌ FAIL: Uplift < +3%")

    # 2. D1 retention check (tiered)
    print()
    print(f"2. D1 Retention: {d1_test['cohort_prop']:.2f}% vs {d1_test['control_prop']:.2f}%")
    print(f"   Delta: {d1_test['delta_pp']:+.2f} pp (p={d1_test['p_value']:.4f})")

    if not d1_test['significant']:
        d1_pass = True
        print(f"   ✅ PASS: Not significant → no proven retention impact")
    else:
        # Significant change: apply strict threshold
        d1_pass = abs(d1_test['delta_pp']) <= 1.0
        if d1_pass:
            print(f"   ✅ PASS: Significant but within ±1 pp tolerance")
        else:
            print(f"   ❌ FAIL: Significant drop > 1 pp")

    # 3. Avg session length check (tiered)
    print()
    print(f"3. Avg Session Length: {avg_session_test['cohort_mean']:.2f}s vs {avg_session_test['control_mean']:.2f}s")
    print(f"   Delta: {avg_session_test['delta_pct']:+.2f}% (p={avg_session_test['p_value']:.4f})")

    if not avg_session_test['significant']:
        avg_session_pass = True
        print(f"   ✅ PASS: Not significant → no proven engagement impact")
    else:
        # Significant change: apply strict 3% threshold
        avg_session_pass = avg_session_test['delta_pct'] >= -3.0
        if avg_session_pass:
            print(f"   ✅ PASS: Significant but drop ≤ 3%")
        else:
            print(f"   ❌ FAIL: Significant drop > 3% ({avg_session_test['delta_pct']:.2f}%)")

    # 4. Median session length check (tiered)
    print()
    print(f"4. Median Session Length: {median_session_test['cohort_median']:.2f}s vs {median_session_test['control_median']:.2f}s")
    print(f"   Delta: {median_session_test['delta_pct']:+.2f}% (p={median_session_test['p_value']:.4f})")

    if not median_session_test['significant']:
        median_session_pass = True
        print(f"   ✅ PASS: Not significant → no proven engagement impact")
    else:
        # Significant change: apply strict 4% threshold
        median_session_pass = median_session_test['delta_pct'] >= -4.0
        if median_session_pass:
            print(f"   ✅ PASS: Significant but drop ≤ 4%")
        else:
            print(f"   ❌ FAIL: Significant drop > 4% ({median_session_test['delta_pct']:.2f}%)")

    # Overall verdict
    print()
    all_pass = arpu_pass and d1_pass and avg_session_pass and median_session_pass
    if all_pass:
        print(f"🏆 VERDICT: {cohort} PASSES all criteria globally")
    else:
        print(f"❌ VERDICT: {cohort} FAILS global criteria")
        failed_criteria = []
        if not arpu_pass: failed_criteria.append("ARPU")
        if not d1_pass: failed_criteria.append("D1")
        if not avg_session_pass: failed_criteria.append("Avg Session")
        if not median_session_pass: failed_criteria.append("Median Session")
        print(f"   Failed on: {', '.join(failed_criteria)}")

    print()
    print("=" * 80)
    print()

    # Store results
    global_results.append({
        'cohort': cohort,
        'arpu_pass': arpu_pass,
        'd1_pass': d1_pass,
        'avg_session_pass': avg_session_pass,
        'median_session_pass': median_session_pass,
        'overall_pass': all_pass,
        'arpu_delta_pct': arpu_test['delta_pct'],
        'arpu_pvalue': arpu_test['p_value']
    })

# Summary table
print("GLOBAL SUMMARY:")
print("-" * 80)
df_global_summary = pd.DataFrame(global_results)
print(df_global_summary[['cohort', 'arpu_pass', 'd1_pass', 'avg_session_pass',
                          'median_session_pass', 'overall_pass']].to_string(index=False))
print()

global_winners = [r for r in global_results if r['overall_pass']]
if global_winners:
    best_global = max(global_winners, key=lambda x: x['arpu_delta_pct'])
    sig_note = " (borderline significance)" if best_global['arpu_pvalue'] >= 0.05 else ""
    print(f"🏆 GLOBAL WINNER: {best_global['cohort']} (ARPU +{best_global['arpu_delta_pct']:.2f}%{sig_note})")
else:
    print("❌ NO GLOBAL WINNER: Keep control")
print()



GLOBAL EVALUATION

Evaluating xxHigh vs Control:
--------------------------------------------------------------------------------
1. ARPU: 0.149581 vs 0.143681
   Delta: +4.11% (p=0.0197)
   ✅ PASS: Uplift ≥ +3% and p<0.05

2. D1 Retention: 32.59% vs 32.77%
   Delta: -0.18 pp (p=0.7264)
   ✅ PASS: Not significant → no proven retention impact

3. Avg Session Length: 340.24s vs 351.60s
   Delta: -3.23% (p=0.0018)
   ❌ FAIL: Significant drop > 3% (-3.23%)

4. Median Session Length: 170.00s vs 179.00s
   Delta: -5.03% (p=0.0023)
   ❌ FAIL: Significant drop > 4% (-5.03%)

❌ VERDICT: xxHigh FAILS global criteria
   Failed on: Avg Session, Median Session


Evaluating xHigh vs Control:
--------------------------------------------------------------------------------
1. ARPU: 0.148212 vs 0.143681
   Delta: +3.15% (p=0.0700)
   ✅ PASS: Uplift ≥ +3% and 0.05≤p<0.10

2. D1 Retention: 32.32% vs 32.77%
   Delta: -0.44 pp (p=0.3872)
   ✅ PASS: Not significant → no proven retention impact

3. Avg Sess

In [21]:
# ============================================
# 7B. PLATFORM-SPECIFIC EVALUATION
# ============================================

print("=" * 80)
print("PLATFORM-SPECIFIC EVALUATION")
print("=" * 80)
print()

platforms = ['android', 'ios']
platform_results = []

for platform in platforms:
    print(f"{'='*80}")
    print(f"PLATFORM: {platform.upper()}")
    print(f"{'='*80}")
    print()

    for cohort in ['xxHigh', 'xHigh', 'gameTune']:
        print(f"Evaluating {cohort} on {platform}:")
        print("-" * 80)

        # Get tests
        arpu_test = ttest_cohort_vs_control(df, 'total_revenue', cohort, platform=platform)
        d1_test = proptest_cohort_vs_control(df, cohort, platform=platform)
        avg_session_test = ttest_cohort_vs_control(df, 'avg_session_length', cohort, platform=platform)
        median_session_test = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort, platform=platform)

        # Apply tiered checks
        arpu_pass = (arpu_test['p_value'] < 0.10) and (arpu_test['delta_pct'] >= 3.0)

        if not d1_test['significant']:
            d1_pass = True
        else:
            d1_pass = abs(d1_test['delta_pp']) <= 1.0

        if not avg_session_test['significant']:
            avg_session_pass = True
        else:
            avg_session_pass = avg_session_test['delta_pct'] >= -3.0

        if not median_session_test['significant']:
            median_session_pass = True
        else:
            median_session_pass = median_session_test['delta_pct'] >= -4.0

        all_pass = arpu_pass and d1_pass and avg_session_pass and median_session_pass

        # Display
        print(f"  ARPU: {arpu_test['delta_pct']:+.2f}% (p={arpu_test['p_value']:.4f}) {'✅' if arpu_pass else '❌'}")
        print(f"  D1: {d1_test['delta_pp']:+.2f} pp (p={d1_test['p_value']:.4f}) {'✅' if d1_pass else '❌'}")

        avg_note = "" if not avg_session_test['significant'] else f" [sig, strict rule]"
        print(f"  Avg Session: {avg_session_test['delta_pct']:+.2f}% (p={avg_session_test['p_value']:.4f}){avg_note} {'✅' if avg_session_pass else '❌'}")

        median_note = "" if not median_session_test['significant'] else f" [sig, strict rule]"
        print(f"  Median Session: {median_session_test['delta_pct']:+.2f}% (p={median_session_test['p_value']:.4f}){median_note} {'✅' if median_session_pass else '❌'}")

        print(f"  → {'🏆 PASS' if all_pass else '❌ FAIL'}")

        if not all_pass:
            failed = []
            if not arpu_pass: failed.append("ARPU")
            if not d1_pass: failed.append("D1")
            if not avg_session_pass: failed.append("Avg Session")
            if not median_session_pass: failed.append("Median Session")
            print(f"     Failed: {', '.join(failed)}")

        print()

        platform_results.append({
            'platform': platform,
            'cohort': cohort,
            'arpu_delta_pct': arpu_test['delta_pct'],
            'overall_pass': all_pass
        })

    # Platform winner
    platform_winners = [r for r in platform_results if r['platform'] == platform and r['overall_pass']]
    if platform_winners:
        best = max(platform_winners, key=lambda x: x['arpu_delta_pct'])
        print(f"🏆 {platform.upper()} WINNER: {best['cohort']} (ARPU +{best['arpu_delta_pct']:.2f}%)")
    else:
        print(f"❌ {platform.upper()}: Keep control (no variant passes all criteria)")
    print()

print("=" * 80)
print()

PLATFORM-SPECIFIC EVALUATION

PLATFORM: ANDROID

Evaluating xxHigh on android:
--------------------------------------------------------------------------------
  ARPU: +4.96% (p=0.0155) ✅
  D1: +0.43 pp (p=0.4768) ✅
  Avg Session: -2.44% (p=0.0469) [sig, strict rule] ✅
  Median Session: -5.13% (p=0.0385) [sig, strict rule] ❌
  → ❌ FAIL
     Failed: Median Session

Evaluating xHigh on android:
--------------------------------------------------------------------------------
  ARPU: +1.56% (p=0.4235) ❌
  D1: -0.88 pp (p=0.1431) ✅
  Avg Session: +1.57% (p=0.2128) ✅
  Median Session: -1.03% (p=0.9594) ✅
  → ❌ FAIL
     Failed: ARPU

Evaluating gameTune on android:
--------------------------------------------------------------------------------
  ARPU: -0.11% (p=0.9563) ❌
  D1: -0.68 pp (p=0.2577) ✅
  Avg Session: -1.72% (p=0.1603) ✅
  Median Session: -3.59% (p=0.1489) ✅
  → ❌ FAIL
     Failed: ARPU

❌ ANDROID: Keep control (no variant passes all criteria)

PLATFORM: IOS

Evaluating xxHigh o

In [22]:
# ============================================
# 7C. REGION-SPECIFIC EVALUATION (All Regions)
# ============================================

print("=" * 80)
print("REGION-SPECIFIC EVALUATION (All Regions)")
print("=" * 80)
print()

# Get all regions except 'Other'
all_regions = [r for r in df['region'].unique() if pd.notnull(r) and r != 'Other']
region_results = []

for region in all_regions:
    print(f"Region: {region}")
    print("-" * 80)

    # Test xxHigh, xHigh and gameTune
    for cohort in ['xxHigh', 'xHigh', 'gameTune']:
        arpu_test = ttest_cohort_vs_control(df, 'total_revenue', cohort, region=region)
        d1_test = proptest_cohort_vs_control(df, cohort, region=region)
        avg_session_test = ttest_cohort_vs_control(df, 'avg_session_length', cohort, region=region)
        median_session_test = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort, region=region)

        if arpu_test and d1_test:
            # Apply tiered thresholds
            arpu_pass = (arpu_test['p_value'] < 0.10) and (arpu_test['delta_pct'] >= 3.0)

            if not d1_test['significant']:
                d1_pass = True
            else:
                d1_pass = abs(d1_test['delta_pp']) <= 1.0

            if not avg_session_test['significant']:
                avg_session_pass = True
            else:
                avg_session_pass = avg_session_test['delta_pct'] >= -3.0

            if not median_session_test['significant']:
                median_session_pass = True
            else:
                median_session_pass = median_session_test['delta_pct'] >= -4.0

            all_pass = arpu_pass and d1_pass and avg_session_pass and median_session_pass

            print(f"  {cohort}:")
            print(f"    ARPU: {arpu_test['delta_pct']:+.2f}% (p={arpu_test['p_value']:.4f}) {'✅' if arpu_pass else '❌'}")
            print(f"    D1: {d1_test['delta_pp']:+.2f} pp (p={d1_test['p_value']:.4f}) {'✅' if d1_pass else '❌'}")

            avg_note = " [sig]" if avg_session_test['significant'] else ""
            print(f"    Avg Session: {avg_session_test['delta_pct']:+.2f}% (p={avg_session_test['p_value']:.4f}){avg_note} {'✅' if avg_session_pass else '❌'}")

            median_note = " [sig]" if median_session_test['significant'] else ""
            print(f"    Median Session: {median_session_test['delta_pct']:+.2f}% (p={median_session_test['p_value']:.4f}){median_note} {'✅' if median_session_pass else '❌'}")

            print(f"    → {'✅ PASS' if all_pass else '❌ FAIL'}")

            if not all_pass:
                failed = []
                if not arpu_pass: failed.append("ARPU")
                if not d1_pass: failed.append("D1")
                if not avg_session_pass: failed.append("Avg Session")
                if not median_session_pass: failed.append("Median Session")
                print(f"       Failed: {', '.join(failed)}")

            region_results.append({
                'region': region,
                'cohort': cohort,
                'overall_pass': all_pass,
                'arpu_delta_pct': arpu_test['delta_pct']
            })
        else:
            print(f"  {cohort}: Insufficient data")

    # Region winner
    region_winners = [r for r in region_results if r['region'] == region and r['overall_pass']]
    if region_winners:
        best = max(region_winners, key=lambda x: x['arpu_delta_pct'])
        print(f"  🏆 Winner: {best['cohort']} (ARPU +{best['arpu_delta_pct']:.2f}%)")
    else:
        print(f"  ❌ Recommendation: Keep control")

    print()

print("=" * 80)
print()


REGION-SPECIFIC EVALUATION (All Regions)

Region: Asia-Pacific
--------------------------------------------------------------------------------
  xxHigh:
    ARPU: +2.75% (p=0.7104) ❌
    D1: +0.81 pp (p=0.6689) ✅
    Avg Session: -7.01% (p=0.0559) ✅
    Median Session: -8.11% (p=0.0627) ✅
    → ❌ FAIL
       Failed: ARPU
  xHigh:
    ARPU: -4.86% (p=0.4835) ❌
    D1: -0.09 pp (p=0.9604) ✅
    Avg Session: -2.62% (p=0.4746) ✅
    Median Session: -4.32% (p=0.4553) ✅
    → ❌ FAIL
       Failed: ARPU
  gameTune:
    ARPU: -10.24% (p=0.1221) ❌
    D1: +2.99 pp (p=0.1182) ✅
    Avg Session: -8.29% (p=0.0198) [sig] ❌
    Median Session: -9.73% (p=0.0255) [sig] ❌
    → ❌ FAIL
       Failed: ARPU, Avg Session, Median Session
  ❌ Recommendation: Keep control

Region: LATAM
--------------------------------------------------------------------------------
  xxHigh:
    ARPU: +1.88% (p=0.8509) ❌
    D1: +2.44 pp (p=0.2068) ✅
    Avg Session: +0.94% (p=0.8003) ✅
    Median Session: +1.14% (p=0.7209)

In [23]:
# ============================================
# 7D. FINAL RECOMMENDATION TABLE
# ============================================

print("=" * 80)
print("FINAL RECOMMENDATIONS")
print("=" * 80)
print()

recommendations = []

# Global
if global_winners:
    best = max(global_winners, key=lambda x: x['arpu_delta_pct'])
    sig_note = " (borderline sig)" if best['arpu_pvalue'] >= 0.05 else ""
    recommendations.append({
        'Segment': 'Global',
        'Recommendation': best['cohort'],
        'ARPU_Uplift': f"+{best['arpu_delta_pct']:.2f}%{sig_note}",
        'Rationale': 'Passes all tiered criteria'
    })
else:
    recommendations.append({
        'Segment': 'Global',
        'Recommendation': 'control',
        'ARPU_Uplift': '—',
        'Rationale': 'No variant passes strict guardrails'
    })

# Platform-specific
for platform in platforms:
    platform_winners_filtered = [r for r in platform_results if r['platform'] == platform and r['overall_pass']]
    if platform_winners_filtered:
        best = max(platform_winners_filtered, key=lambda x: x['arpu_delta_pct'])
        recommendations.append({
            'Segment': f'{platform.capitalize()}',
            'Recommendation': best['cohort'],
            'ARPU_Uplift': f"+{best['arpu_delta_pct']:.2f}%",
            'Rationale': f'Best on {platform} with acceptable guardrails'
        })
    else:
        recommendations.append({
            'Segment': f'{platform.capitalize()}',
            'Recommendation': 'control',
            'ARPU_Uplift': '—',
            'Rationale': 'No variant passes tiered criteria'
        })

# All regions
for region in all_regions:
    region_winners_filtered = [r for r in region_results if r['region'] == region and r['overall_pass']]
    if region_winners_filtered:
        best = max(region_winners_filtered, key=lambda x: x['arpu_delta_pct'])
        recommendations.append({
            'Segment': region,
            'Recommendation': best['cohort'],
            'ARPU_Uplift': f"+{best['arpu_delta_pct']:.2f}%",
            'Rationale': 'Passes tiered criteria'
        })
    else:
        recommendations.append({
            'Segment': region,
            'Recommendation': 'control',
            'ARPU_Uplift': '—',
            'Rationale': 'Engagement cost too high or no uplift'
        })

# gameTune overall verdict
gametune_result = [r for r in global_results if r['cohort'] == 'gameTune'][0]
recommendations.append({
    'Segment': '─────────────',
    'Recommendation': '─────────────',
    'ARPU_Uplift': '─────────────',
    'Rationale': '─────────────'
})
recommendations.append({
    'Segment': 'gameTune (Overall)',
    'Recommendation': 'NOT RECOMMENDED',
    'ARPU_Uplift': f"{gametune_result['arpu_delta_pct']:+.2f}%",
    'Rationale': 'No monetization advantage over fixed rules, adds complexity'
})

df_recommendations = pd.DataFrame(recommendations)
print(df_recommendations.to_string(index=False))
print()

print("=" * 80)
print("KEY INSIGHTS:")
print("  • Tiered thresholds correctly penalize variants with *proven*")
print("    engagement damage (significant drops with strict limits)")
print("  • Non-significant changes automatically pass (no proven harm)")
print("  • Platform and regional analysis reveals differentiated strategies")
print("=" * 80)
print()
print("ANALYSIS COMPLETE")
print("=" * 80)


FINAL RECOMMENDATIONS

                   Segment  Recommendation             ARPU_Uplift                                                   Rationale
                    Global           xHigh +3.15% (borderline sig)                                  Passes all tiered criteria
                   Android         control                       —                           No variant passes tiered criteria
                       Ios           xHigh                  +6.69%                      Best on ios with acceptable guardrails
              Asia-Pacific         control                       —                       Engagement cost too high or no uplift
                     LATAM         control                       —                       Engagement cost too high or no uplift
             North America         control                       —                       Engagement cost too high or no uplift
                    Europe           xHigh                  +7.38%                      

In [24]:
# ============================================
# Exporting all analysis results for Tableau
# ============================================

# 1. Global results
df_global_export = pd.DataFrame(global_results)
df_global_export.to_csv('global_results.csv', index=False)

# 2. Platform results
df_platform_export = pd.DataFrame(platform_results)
df_platform_export.to_csv('platform_results.csv', index=False)

# 3. Region results
df_region_export = pd.DataFrame(region_results)
df_region_export.to_csv('region_results.csv', index=False)

# 4. Final recommendations
df_recommendations.to_csv('final_recommendations.csv', index=False)

# 5. ARPU global (with all stats)
arpu_global_detailed = []
for cohort in ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']:
    result = ttest_cohort_vs_control(df, 'total_revenue', cohort)
    if result:
        arpu_global_detailed.append({
            'cohort': cohort,
            'control_arpu': result['control_mean'],
            'cohort_arpu': result['cohort_mean'],
            'delta': result['delta'],
            'delta_pct': result['delta_pct'],
            'p_value': result['p_value'],
            'significant': result['significant'],
            'ci_lower': result['ci_lower'],
            'ci_upper': result['ci_upper']
        })
pd.DataFrame(arpu_global_detailed).to_csv('arpu_global_detailed.csv', index=False)

# 6. D1 global (with all stats)
d1_global_detailed = []
for cohort in ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']:
    result = proptest_cohort_vs_control(df, cohort)
    if result:
        d1_global_detailed.append({
            'cohort': cohort,
            'control_d1': result['control_prop'],
            'cohort_d1': result['cohort_prop'],
            'delta_pp': result['delta_pp'],
            'p_value': result['p_value'],
            'significant': result['significant']
        })
pd.DataFrame(d1_global_detailed).to_csv('d1_global_detailed.csv', index=False)

# 7. Session length global
session_global_detailed = []
for cohort in ['xxHigh', 'xHigh', 'gameTune', 'xLow', 'xxLow']:
    avg_result = ttest_cohort_vs_control(df, 'avg_session_length', cohort)
    median_result = mannwhitney_cohort_vs_control(df, 'median_session_length', cohort)
    if avg_result and median_result:
        session_global_detailed.append({
            'cohort': cohort,
            'avg_control': avg_result['control_mean'],
            'avg_cohort': avg_result['cohort_mean'],
            'avg_delta_pct': avg_result['delta_pct'],
            'avg_pvalue': avg_result['p_value'],
            'avg_significant': avg_result['significant'],
            'median_control': median_result['control_median'],
            'median_cohort': median_result['cohort_median'],
            'median_delta_pct': median_result['delta_pct'],
            'median_pvalue': median_result['p_value'],
            'median_significant': median_result['significant']
        })
pd.DataFrame(session_global_detailed).to_csv('session_global_detailed.csv', index=False)

# 8. Platform detailed (all metrics)
platform_detailed = []
for platform in ['android', 'ios']:
    for cohort in ['xxHigh', 'xHigh', 'gameTune']:
        arpu = ttest_cohort_vs_control(df, 'total_revenue', cohort, platform=platform)
        d1 = proptest_cohort_vs_control(df, cohort, platform=platform)
        avg_sess = ttest_cohort_vs_control(df, 'avg_session_length', cohort, platform=platform)
        if arpu and d1 and avg_sess:
            platform_detailed.append({
                'platform': platform,
                'cohort': cohort,
                'arpu_delta_pct': arpu['delta_pct'],
                'arpu_pvalue': arpu['p_value'],
                'arpu_significant': arpu['significant'],
                'd1_delta_pp': d1['delta_pp'],
                'd1_pvalue': d1['p_value'],
                'd1_significant': d1['significant'],
                'avg_session_delta_pct': avg_sess['delta_pct'],
                'avg_session_pvalue': avg_sess['p_value'],
                'avg_session_significant': avg_sess['significant']
            })
pd.DataFrame(platform_detailed).to_csv('platform_detailed.csv', index=False)

# 9. Region detailed (all metrics)
region_detailed = []
regions = [r for r in df['region'].unique() if pd.notnull(r) and r != 'Other']
for region in regions:
    for cohort in ['xxHigh', 'xHigh', 'gameTune']:
        arpu = ttest_cohort_vs_control(df, 'total_revenue', cohort, region=region)
        d1 = proptest_cohort_vs_control(df, cohort, region=region)
        if arpu and d1:
            region_detailed.append({
                'region': region,
                'cohort': cohort,
                'arpu_delta_pct': arpu['delta_pct'],
                'arpu_pvalue': arpu['p_value'],
                'arpu_significant': arpu['significant'],
                'd1_delta_pp': d1['delta_pp'],
                'd1_pvalue': d1['p_value'],
                'd1_significant': d1['significant']
            })
pd.DataFrame(region_detailed).to_csv('region_detailed.csv', index=False)

print("All CSV files exported successfully!")


All CSV files exported successfully!
